In [0]:
from pyspark.sql.functions import current_timestamp, lit

# ==========================================
# JOB PARAMETERS
# ==========================================

dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("load_type", "incremental")

environment = dbutils.widgets.get("environment")
load_type = dbutils.widgets.get("load_type")

print(f"Environment : {environment}")
print(f"Load Type   : {load_type}")

# ==========================================
# 01 - MORTGAGE GENERIC INGESTION
# ==========================================

config_table = "mortgage_catalog.metadata.ingestion_config"

configs = (
    spark.table(config_table)
    .filter("is_active = true")
    .collect()
)

for config in configs:

    source_name   = config["source_name"]
    source_type   = config["source_type"]
    source_path   = config["source_path"]
    file_format   = config["file_format"]
    delimiter     = config["delimiter"]
    target_schema = config["target_schema"]
    target_table  = config["target_table"]

    target = f"mortgage_catalog.{target_schema}.{target_table}"

    print(f"\nProcessing: {source_name}")
    print(f"Source: {source_path}")
    print(f"Target: {target}")

    # -------------------------------
    # CSV
    # -------------------------------
    if file_format.upper() == "CSV":

        reader = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
        )

        if delimiter:
            reader = reader.option("delimiter", delimiter)

        df = reader.csv(source_path)

    # -------------------------------
    # JSON
    # -------------------------------
    elif file_format.upper() == "JSON":

        df = (
            spark.read
            .option("inferSchema", True)
            .json(source_path)
        )

    # -------------------------------
    # PARQUET
    # -------------------------------
    elif file_format.upper() == "PARQUET":

        df = spark.read.parquet(source_path)

    else:
        print(f"Unsupported format: {file_format}")
        continue

    # -------------------------------
    # Add audit columns
    # -------------------------------

    df = (
        df
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_system", lit(source_name))
        .withColumn("_source_type", lit(source_type))
    )

    # -------------------------------
    # Write Bronze Delta table
    # -------------------------------

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    print(f"SUCCESS → {target}")

print("\nALL INGESTION COMPLETED ✅")

Environment : dev
Load Type   : incremental

Processing: CUSTOMER_FILE
Source: /Volumes/mortgage_catalog/bronze/mortgage_files/landing/customer
Target: mortgage_catalog.bronze.customer_bronze
SUCCESS → mortgage_catalog.bronze.customer_bronze

Processing: PROPERTY_FILE
Source: /Volumes/mortgage_catalog/bronze/mortgage_files/landing/property
Target: mortgage_catalog.bronze.property_bronze
SUCCESS → mortgage_catalog.bronze.property_bronze

Processing: BRANCH_FILE
Source: /Volumes/mortgage_catalog/bronze/mortgage_files/landing/branch
Target: mortgage_catalog.bronze.branch_bronze
SUCCESS → mortgage_catalog.bronze.branch_bronze

Processing: DOCUMENT_FILE
Source: /Volumes/mortgage_catalog/bronze/mortgage_files/landing/document
Target: mortgage_catalog.bronze.document_bronze
SUCCESS → mortgage_catalog.bronze.document_bronze

Processing: AZURE_SQL_CUSTOMER
Source: /Volumes/mortgage_catalog/bronze/mortgage_files/landing/azure_sql/customer
Target: mortgage_catalog.bronze.sql_customer_bronze
SUCCE